# Solution A2 · K-fold, stratified K-fold and `cross_val_score` from scratch

**Type**: Implement from scratch

## Objective

Implement the generation of folds (identical to scikit-learn's `KFold`), a stratified version and a
`my_cross_val_score` function, and verify that with the same folds they produce the same scores.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y = df["defaulted"].to_numpy()
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

## Commented solution

In [ ]:
def kfold_indices(n, k, seed):
    idx = np.arange(n)
    np.random.RandomState(seed).shuffle(idx)
    size = np.full(k, n // k)
    size[: n % k] += 1                                  # the first n % k folds have one more element
    pos = np.cumsum(np.r_[0, size])
    folds = []
    for i in range(k):
        val = np.sort(idx[pos[i]:pos[i + 1]])
        folds.append((np.setdiff1d(np.arange(n), val), val))
    return folds


def strat_indices(y, k, seed):
    r = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        ind = np.flatnonzero(y == c)
        r.shuffle(ind)
        fold_of[ind] = np.arange(len(ind)) % k          # in turn, class by class
    return [(np.flatnonzero(fold_of != i), np.flatnonzero(fold_of == i)) for i in range(k)]


def my_cross_val_score(model, X, y, folds):
    scores = []
    for tr, va in folds:
        m = clone(model).fit(X[tr], y[tr])              # from scratch on every fold: no memory
        scores.append(roc_auc_score(y[va], m.predict_proba(X[va])[:, 1]))
    return np.array(scores)

In [ ]:
folds = kfold_indices(len(y), 5, 0)
my_scores = my_cross_val_score(model, X, y, folds)
sk_scores = cross_val_score(model, X, y, cv=folds, scoring="roc_auc")
print("mine        :", my_scores.round(4))
print("scikit-learn:", sk_scores.round(4))
sf = strat_indices(y, 5, 0)
print("positives per fold (stratified):", [int(y[v].sum()) for _, v in sf], "| (plain K-fold):", [int(y[v].sum()) for _, v in folds])

## Check

In [ ]:
with verify():
    ref = list(KFold(5, shuffle=True, random_state=0).split(np.arange(len(y))))
    check(len(folds) == 5 and all(np.array_equal(a[1], b[1]) and np.array_equal(a[0], b[0]) for a, b in zip(folds, ref)),
          "kfold_indices matches scikit-learn's KFold(shuffle=True)", "same shuffle (RandomState(seed)), folds of size n//k with the remainder on the first ones")
    everyone = np.sort(np.concatenate([v for _, v in folds]))
    check(np.array_equal(everyone, np.arange(len(y))), "the validation folds partition the data", "every row in exactly one validation fold")
    sf = strat_indices(y, 5, 0)
    counts = np.array([[(y[v] == c).sum() for c in (0, 1)] for _, v in sf])
    check(np.all(counts.max(axis=0) - counts.min(axis=0) <= 1), "stratification balances every class across the folds (difference at most 1)",
          "for every class, the counts in the folds must differ by at most 1")
    check(all(len(np.intersect1d(t, v)) == 0 and len(t) + len(v) == len(y) for t, v in sf), "training and validation are disjoint and complete",
          "every pair must partition all the rows")
    check(close(my_scores, sk_scores), "my_cross_val_score matches cross_val_score on the same folds",
          "clone the model at every fold, train on the training part, AUC on the validation fold")
done()

## What you learned

- A cross-validation is a four-line loop: partition, clone the model, train on the training set, evaluate on the rest. There is no other
  magic; the value lies in the choice of **how** to partition (random, stratified, grouped, temporal).
- `clone` is essential: every fold must start from a model with no state. Reusing the same object would make fold 2 learn starting
  from fold 1 (a subtle form of leakage).
- Stratification guarantees the class proportions in every fold: with a rare class it avoids folds with no positives, where the AUC is not
  even defined.

## Common pitfall

Not shuffling the data (`shuffle=False`) when the rows are sorted by class, by time or by group: the folds become contiguous blocks
with different distributions. It is the bug of exercise E3.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why must the model be cloned and not reused from one fold to the next in cross-validation?</summary>

Because `fit` does not reset the state of models that train incrementally (initial coefficients with `warm_start`, some models
with `partial_fit`), and even where it does reset it the cost of getting it wrong is silent. Reusing a model already trained on other folds
contaminates the evaluation with information from the past. `clone` creates a model with the same hyperparameters and no state.

</details>